# E3 producer — GNN training runs (no analysis in this notebook)

Trains the comparison models and saves everything; the tables, probes,
and QuIC comparison live in a separate analysis consumer. GPU session.

Design decisions on record:
- Four arms. GIN-const and GCN-const: 1-WL on regular graphs with
  uniform features is the degenerate case — the theory says these
  collapse to a constant representation. GIN-RNI and GIN-LapPE: the two
  cheap fixes a practitioner would reach for, given their best shot.
- RNI at full strength: features resampled every epoch; eval predictions
  and embeddings averaged over 10 draws.
- LapPE: k=8 Laplacian eigenvectors (nontrivial ones), random sign flips
  as train-time augmentation. Note for the analysis: LapPE is
  spectrum-derived, so cospectral mates are indistinguishable to it by
  construction.
- Collapse is MEASURED, not asserted: max pairwise embedding distance
  across the enumeration, reported per trained model for all four arms.
  We have theories about this object; the holes aren't all plugged.
- Honest-effort tuning: depth x lr grid (4 points), width 64, selected
  on the fold-0/seed-0 validation split per (n, model, target), reused
  across folds and seeds. Sum pooling (mean pooling on regular graphs
  is an extra handicap we don't hand them). Early stopping on val.
- Implementation: pure-torch dense GIN/GCN. Graphs are 14/16-node and
  regular; dense full-batch is exact, fast, and removes the PyG
  install/version failure mode (internet stays off).
- Outer folds: KFold(5, shuffle, seed=0) — identical to the E2 splits.
  Targets z-scored on train; R2 reported on raw counts; negative R2
  saved as-is.
- SESSION KNOB: RUN_NS / RUN_MODELS below — split across parallel
  sessions freely; each (n, model) saves its own pkl on completion.

In [1]:
import pickle
import time

import numpy as np
import networkx as nx
import torch
import torch.nn as nn

from numpy.linalg import eigvalsh, eigh
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

from tqdm.notebook import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', DEVICE)

torch 2.10.0+cu128 | device: cuda


In [2]:
SEED = 0
N_SPLITS = 5
SEEDS = (0, 1, 2)

# --- session knob: subset for parallel launches ---
RUN_NS = (14, 16)
RUN_MODELS = ('gin_const', 'gcn_const', 'gin_rni', 'gin_lappe')

# --- honest-effort tuning grid ---
DEPTHS = (3, 5)
LRS = (1e-3, 1e-2)
WIDTH = 64
MAX_EPOCHS = 300
PATIENCE = 30
VAL_FRAC = 0.15

RNI_DIM = 8
RNI_EVAL_DRAWS = 10
LAPPE_K = 8

BASES = {
    14: "/kaggle/input/notebooks/lukemiller1987/n14-aaai-dataset/",
    16: "/kaggle/input/notebooks/lukemiller1987/n16-aaai-dataset/",
}
EXPECTED_CUBIC_COUNTS = {14: 509, 16: 4060}

In [3]:
def load_dataset(n):
    with open(BASES[n] + f"n{n}_data_dict.pkl", 'rb') as f:
        data_dict = pickle.load(f)
    keys = sorted(data_dict)
    assert len(keys) == EXPECTED_CUBIC_COUNTS[n], (
        f'expected {EXPECTED_CUBIC_COUNTS[n]} graphs at n={n}, got {len(keys)}')

    graphs = {k: nx.from_graph6_bytes(data_dict[k]['graph6'].encode())
              for k in keys}
    A = np.stack([data_dict[k]['adj_mat'] for k in keys]).astype(np.float32)

    def field_or(field, fn, progress=False):
        if field in data_dict[keys[0]]:
            return np.array([data_dict[k][field] for k in keys], dtype=float)
        it = tqdm(keys, desc=f'n={n} {field}') if progress else keys
        return np.array([fn(k) for k in it], dtype=float)

    def c6_count(k):
        return sum(1 for c in nx.simple_cycles(graphs[k], length_bound=6)
                   if len(c) == 6)

    spectra = (np.stack([data_dict[k]['spectrum'] for k in keys])
               if 'spectrum' in data_dict[keys[0]]
               else np.stack([np.sort(eigvalsh(data_dict[k]['adj_mat']))
                              for k in keys]))

    targets = {
        'triangles':    field_or('num_triangles', None),
        '4-cycles':     field_or('num_4_cycles', None),
        '5-cycles':     field_or('num_5_cycles', None),
        '6-cycles':     field_or('num_6_cycles', c6_count, progress=True),
        'girth':        field_or('girth',    lambda k: nx.girth(graphs[k])),
        'diameter':     field_or('diameter', lambda k: nx.diameter(graphs[k])),
        'spectral_gap': spectra[:, -1] - spectra[:, -2],
    }

    # LapPE: eigenvectors of L = D - A; regular => same eigvecs as A.
    # Drop the trivial constant eigenvector, take the next LAPPE_K.
    lap_pe = np.zeros((len(keys), n, LAPPE_K), dtype=np.float32)
    for i, k in enumerate(keys):
        L = np.diag(A[i].sum(1)) - A[i]
        w, v = eigh(L)
        lap_pe[i] = v[:, 1:1 + LAPPE_K]

    return {'A': A, 'targets': targets, 'lap_pe': lap_pe, 'n_nodes': n}

DATA = {n: load_dataset(n) for n in RUN_NS}
for n in RUN_NS:
    print(f"n={n}: A {DATA[n]['A'].shape}, lap_pe {DATA[n]['lap_pe'].shape}")

n=14 num_6_cycles:   0%|          | 0/509 [00:00<?, ?it/s]

n=16 num_6_cycles:   0%|          | 0/4060 [00:00<?, ?it/s]

n=14: A (509, 14, 14), lap_pe (509, 14, 8)
n=16: A (4060, 16, 16), lap_pe (4060, 16, 8)


In [4]:
SPLITS = {}
for n in RUN_NS:
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    SPLITS[n] = [(tr, te) for tr, te in kf.split(DATA[n]['A'])]

## Models — dense GIN / GCN, sum pooling

`embedding` = sum-pooled node representations after the last conv
(width = WIDTH); the head is a 2-layer MLP on top. The pooled embedding
is what gets saved for the analysis consumer's linear probe.

In [5]:
class DenseGIN(nn.Module):
    def __init__(self, in_dim, width, depth):
        super().__init__()
        self.eps = nn.Parameter(torch.zeros(depth))
        dims = [in_dim] + [width] * depth
        self.mlps = nn.ModuleList([
            nn.Sequential(nn.Linear(dims[i], width), nn.ReLU(),
                          nn.Linear(width, width), nn.ReLU())
            for i in range(depth)])
        self.head = nn.Sequential(nn.Linear(width, width), nn.ReLU(),
                                  nn.Linear(width, 1))

    def forward(self, A, X):
        h = X
        for i, mlp in enumerate(self.mlps):
            h = mlp((1 + self.eps[i]) * h + torch.bmm(A, h))
        emb = h.sum(dim=1)                     # sum pooling -> (B, width)
        return self.head(emb).squeeze(-1), emb


class DenseGCN(nn.Module):
    def __init__(self, in_dim, width, depth):
        super().__init__()
        dims = [in_dim] + [width] * depth
        self.lins = nn.ModuleList([nn.Linear(dims[i], width)
                                   for i in range(depth)])
        self.head = nn.Sequential(nn.Linear(width, width), nn.ReLU(),
                                  nn.Linear(width, 1))

    def forward(self, A_hat, X):
        h = X
        for lin in self.lins:
            h = torch.relu(lin(torch.bmm(A_hat, h)))
        emb = h.sum(dim=1)
        return self.head(emb).squeeze(-1), emb


def gcn_norm(A):
    """D^-1/2 (A+I) D^-1/2, dense batch."""
    B, n, _ = A.shape
    Ai = A + torch.eye(n, device=A.device).expand(B, n, n)
    d = Ai.sum(-1)
    dinv = d.pow(-0.5)
    return dinv.unsqueeze(-1) * Ai * dinv.unsqueeze(1)


def make_features(mode, B, n, lap_pe=None, generator=None, device=DEVICE):
    if mode == 'const':
        return torch.ones(B, n, 1, device=device)
    if mode == 'rni':
        return torch.randn(B, n, RNI_DIM, generator=generator, device=device)
    if mode == 'lappe':
        signs = (torch.randint(0, 2, (B, 1, LAPPE_K), generator=generator,
                               device=device) * 2 - 1).float() \
                if generator is not None else 1.0
        return lap_pe * signs
    raise ValueError(mode)

MODEL_SPECS = {
    'gin_const': ('gin', 'const', 1),
    'gcn_const': ('gcn', 'const', 1),
    'gin_rni':   ('gin', 'rni',   RNI_DIM),
    'gin_lappe': ('gin', 'lappe', LAPPE_K),
}

## Training loop

One function per trained model instance: carve val from train, z-score y
on train, Adam, early stop on val MSE, restore best weights, evaluate on
raw scale, return test R2 + embeddings and predictions for ALL graphs +
the collapse diagnostic (max pairwise embedding L2 distance across the
full enumeration, and its ratio to mean embedding norm). RNI eval
averages predictions and embeddings over RNI_EVAL_DRAWS feature
draws.

In [6]:
RNI_VAL_DRAWS = 5       # val loss on a single draw is noise; average it
RNI_MAX_EPOCHS = 1000   # RNI sees a new input every epoch — needs budget
RNI_PATIENCE = 60


def eval_all(model, A, feats_fn, arch):
    model.eval()
    with torch.no_grad():
        if arch == 'rni':
            preds, embs = [], []
            for _ in range(RNI_EVAL_DRAWS):
                p, e = model(A, feats_fn(train_mode=False, draw=True))
                preds.append(p); embs.append(e)
            return (torch.stack(preds).mean(0),
                    torch.stack(embs).mean(0))
        return model(A, feats_fn(train_mode=False))


def train_one(n, model_name, target_name, seed, fold, depth, lr,
              verbose=False):
    D = DATA[n]
    arch, feat_mode, in_dim = MODEL_SPECS[model_name]
    tr_all, te = SPLITS[n][fold]

    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(tr_all))
    n_val = max(1, int(VAL_FRAC * len(tr_all)))
    val, tr = tr_all[perm[:n_val]], tr_all[perm[n_val:]]

    y_raw = D['targets'][target_name]
    mu, sd = y_raw[tr].mean(), y_raw[tr].std() + 1e-12
    y = torch.tensor((y_raw - mu) / sd, dtype=torch.float32, device=DEVICE)

    A_t = torch.tensor(D['A'], device=DEVICE)
    A_in = gcn_norm(A_t) if arch == 'gcn' else A_t
    lap = (torch.tensor(D['lap_pe'], device=DEVICE)
           if feat_mode == 'lappe' else None)

    gen = torch.Generator(device=DEVICE); gen.manual_seed(seed * 1000 + fold)

    def feats(train_mode=True, draw=False):
        if feat_mode == 'const':
            return make_features('const', len(A_t), D['n_nodes'])
        if feat_mode == 'rni':
            return make_features('rni', len(A_t), D['n_nodes'],
                                 generator=gen)
        return make_features('lappe', len(A_t), D['n_nodes'], lap_pe=lap,
                             generator=gen if train_mode else None)

    torch.manual_seed(seed * 100 + fold)
    Net = DenseGIN if arch == 'gin' else DenseGCN
    model = Net(in_dim, WIDTH, depth).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.MSELoss()

    # RNI gets a larger budget and noise-averaged validation; a single
    # fresh draw per val pass makes early stopping fire on draw luck.
    max_ep   = RNI_MAX_EPOCHS if feat_mode == 'rni' else MAX_EPOCHS
    patience = RNI_PATIENCE   if feat_mode == 'rni' else PATIENCE
    val_draws = RNI_VAL_DRAWS if feat_mode == 'rni' else 1

    best_val, best_state, best_epoch, since = np.inf, None, 0, 0
    for epoch in range(max_ep):
        model.train()
        opt.zero_grad()
        pred, _ = model(A_in[tr], feats()[tr])
        loss = lossf(pred, y[tr])
        loss.backward(); opt.step()

        model.eval()
        with torch.no_grad():
            vl = float(np.mean([
                lossf(model(A_in[val], feats(train_mode=False)[val])[0],
                      y[val]).item()
                for _ in range(val_draws)]))
        if vl < best_val - 1e-6:
            best_val, best_epoch, since = vl, epoch, 0
            best_state = {k: v.detach().clone()
                          for k, v in model.state_dict().items()}
        else:
            since += 1
            if since >= patience:
                break
    model.load_state_dict(best_state)

    pred_all, emb_all = eval_all(model, A_in, feats, feat_mode)
    pred_raw = (pred_all.cpu().numpy() * sd + mu)
    emb_np = emb_all.cpu().numpy().astype(np.float32)

    # collapse diagnostic: measured, not asserted. Exact chunked
    # computation in float64 — cdist's matmul path cancels
    # catastrophically for near-identical points and floors at
    # ~sqrt(eps)*norm, which is noise, not spread.
    e64 = emb_all.double()
    max_pair = 0.0
    for s in range(0, len(e64), 256):
        diff = e64[s:s + 256, None, :] - e64[None, :, :]
        max_pair = max(max_pair, diff.norm(dim=-1).max().item())
    mean_norm = e64.norm(dim=1).mean().item()

    return {
        'test_r2': r2_score(y_raw[te], pred_raw[te]),
        'val_r2':  r2_score(y_raw[val], pred_raw[val]),
        'best_epoch': best_epoch,
        'predictions_raw': pred_raw.astype(np.float32),
        'embeddings': emb_np,
        'collapse_max_pair_dist': max_pair,
        'collapse_mean_norm': mean_norm,
        'collapse_ratio': max_pair / (mean_norm + 1e-12),
        'depth': depth, 'lr': lr, 'seed': seed, 'fold': fold,
    }

## Tuning pass, then the full runs

Grid selected on fold-0 / seed-0 validation R2 per (n, model, target),
reused across all folds and seeds. Each completed (n, model) saves
`n{n}_e3_{model}.pkl` immediately — session death loses at most one
model's runs.

In [7]:
for n in RUN_NS:
    for model_name in RUN_MODELS:
        t0 = time.time()
        results = {'config': {
            'width': WIDTH, 'grid_depths': DEPTHS, 'grid_lrs': LRS,
            'max_epochs': MAX_EPOCHS, 'patience': PATIENCE,
            'val_frac': VAL_FRAC, 'seeds': SEEDS, 'n_splits': N_SPLITS,
            'split_seed': SEED, 'rni_dim': RNI_DIM,
            'rni_eval_draws': RNI_EVAL_DRAWS, 'lappe_k': LAPPE_K,
            'torch': torch.__version__, 'device': DEVICE,
            'splits': [(tr.tolist(), te.tolist()) for tr, te in SPLITS[n]],
            'note': 'splits identical to E2 (KFold 5, shuffle, seed 0)'},
            'selected': {}, 'runs': {}}

        for target in DATA[n]['targets']:
            # tuning on fold 0, seed 0
            best, best_cfg = -np.inf, None
            for depth in DEPTHS:
                for lr in LRS:
                    r = train_one(n, model_name, target, SEEDS[0], 0,
                                  depth, lr)
                    if r['val_r2'] > best:
                        best, best_cfg = r['val_r2'], (depth, lr)
            results['selected'][target] = {'depth': best_cfg[0],
                                           'lr': best_cfg[1],
                                           'val_r2': best}

            runs = []
            for seed in SEEDS:
                for fold in range(N_SPLITS):
                    runs.append(train_one(n, model_name, target, seed,
                                          fold, *best_cfg))
            results['runs'][target] = runs

            r2s = np.array([r['test_r2'] for r in runs])
            col = np.array([r['collapse_ratio'] for r in runs])
            print(f'n={n} {model_name:>10} {target:>13}: '
                  f'R2 = {r2s.mean():+.3f} +- {r2s.std():.3f}   '
                  f'collapse ratio = {col.mean():.2e}   '
                  f'cfg = {best_cfg}')

        with open(f'/kaggle/working/n{n}_e3_{model_name}.pkl', 'wb') as f:
            pickle.dump(results, f)
        print(f'saved n{n}_e3_{model_name}.pkl '
              f'({time.time() - t0:.0f}s)\n')

n=14  gin_const     triangles: R2 = -0.013 +- 0.026   collapse ratio = 0.00e+00   cfg = (3, 0.001)
n=14  gin_const      4-cycles: R2 = -0.036 +- 0.050   collapse ratio = 0.00e+00   cfg = (3, 0.01)
n=14  gin_const      5-cycles: R2 = -0.024 +- 0.024   collapse ratio = 0.00e+00   cfg = (3, 0.001)
n=14  gin_const      6-cycles: R2 = -0.019 +- 0.024   collapse ratio = 0.00e+00   cfg = (5, 0.01)
n=14  gin_const         girth: R2 = -0.015 +- 0.026   collapse ratio = 0.00e+00   cfg = (3, 0.01)
n=14  gin_const      diameter: R2 = -0.023 +- 0.032   collapse ratio = 0.00e+00   cfg = (3, 0.001)
n=14  gin_const  spectral_gap: R2 = -0.012 +- 0.023   collapse ratio = 0.00e+00   cfg = (3, 0.01)
saved n14_e3_gin_const.pkl (41s)

n=14  gcn_const     triangles: R2 = -0.008 +- 0.009   collapse ratio = 0.00e+00   cfg = (3, 0.01)
n=14  gcn_const      4-cycles: R2 = -0.024 +- 0.028   collapse ratio = 0.00e+00   cfg = (3, 0.01)
n=14  gcn_const      5-cycles: R2 = -0.028 +- 0.027   collapse ratio = 0.00e+00  

## Done — producer only

Analysis consumer (separate notebook, after all pkls exist): the
headline table (trained-network test R2 vs QuIC linear probe from
n{n}_e2_ridge_probe.pkl, same folds), linear probes on the saved
embeddings for protocol parallelism, the collapse-diagnostic table for
the const arms, the matched-dimension sentence read off the E7 curve at
k = WIDTH, and the LapPE-vs-cospectral-mates check.